# Third Attempt

In [20]:
import torch
import numpy as np

## Load/ Preprocess Data

In [23]:
import pandas as pd
import numpy as np
import wfdb
import ast

#provided function to load data with information in the agg_df (modified to return a numpy array)
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data]).astype(np.float32)
    return data

# path to dta and selected sampling rate
path = './ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/'
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x)) #changes class distribution to dict

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1] #only take diagnostic scps

classes = agg_df.index.tolist()
sub_classes = agg_df.diagnostic_subclass.unique().tolist()
super_classes = agg_df.diagnostic_class.unique().tolist()

# encodes the labeled as a 44d vector of 1's and zeros (based off of classes list)
def encode_multilabel(class_list):
    return np.array([1 if cls in class_list else 0 for cls in classes])

# encodes the labeled as a 44d vector of 1's and zeros (based off of classes list)
def encode_multilabel_index(class_list):
    return np.array([classes.index(cls) for cls in class_list])

# encodes the labeled as a 23d vector of 1's and zeros (based off of sub classes list)
def encode_multilabel_sub(class_list):
    return np.array([1 if cls in class_list else 0 for cls in sub_classes])

# encodes the labeled as a 5d vector of 1's and zeros (based off of sub classes list)
def encode_multilabel_super(class_list):
    return np.array([1 if cls in class_list else 0 for cls in super_classes])

def aggregate_diagnostic(y_dic):
    tmp = []
    argmaxs = []
    max_prct = 0
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(key) #diagnostic, diagnostic_subclass #agg_df.loc[key].index
            if y_dic[key] > max_prct:
                if argmaxs != []:
                    argmaxs.pop(0)
                argmaxs.append(key)
    return list(set(tmp)), list(set(argmaxs))

def aggregate_diagnostic_max(y_dic):
    tmp = []
    max_prct = 0
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] >= max_prct:
                tmp.append(key)
                max_prct = y_dic[key]
    return list(set(tmp))

def aggregate_diagnostic_sub(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(agg_df.loc[key].diagnostic_subclass)
    return list(set(tmp))

def aggregate_diagnostic_super(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass and encoding multilabel
Y['indv_diagnostic'] = Y.scp_codes.apply(aggregate_diagnostic).apply(encode_multilabel)
Y['indv_diagnostic_str'] = str(Y['indv_diagnostic'])
Y['indv_diagnostic_max'] = Y.scp_codes.apply(aggregate_diagnostic_max).apply(encode_multilabel_index)

Y['sub_diagnostic'] = Y.scp_codes.apply(aggregate_diagnostic_sub).apply(encode_multilabel_sub)
Y['sub_diagnostic_str'] = str(Y['sub_diagnostic'])

Y['super_diagnostic'] = Y.scp_codes.apply(aggregate_diagnostic_super).apply(encode_multilabel_super)
Y['super_diagnostic_str'] = str(Y['super_diagnostic'])




In [26]:
Y['indv_diagnostic_max']

ecg_id
1         [4]
2         [4]
3         [4]
4         [4]
5         [4]
         ... 
21833     [0]
21834     [4]
21835    [25]
21836     [4]
21837     [4]
Name: indv_diagnostic_max, Length: 21837, dtype: object

## Data Prep

In [27]:
Y_matrix = np.stack(Y.indv_diagnostic.values).astype(np.float32)
Y_cond = pd.DataFrame(Y_matrix)
Y_cond = Y_cond.loc[:, Y_cond.any()]

In [28]:
from torch.utils.data import WeightedRandomSampler

condition_freq = Y_cond.sum(axis=0)
condition_inv_freq = 1.0 / (condition_freq + 1e-6)

row_weights = (Y_cond * condition_inv_freq).sum(axis=1)



sampler = WeightedRandomSampler(
    weights=row_weights,
    num_samples=len(row_weights),
    replacement = True
)

In [29]:
pos_weight = (len(Y) - condition_freq) / condition_freq
loss = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))

In [30]:
Y_cond['filename_lr'] = Y['filename_lr'].to_numpy()
Y_cond = Y_cond[~(Y_cond.drop(columns=['filename_lr'])==0).all(axis=1)]
Y_cond

,filename_lr


In [31]:
Y_cond['total_conds'] = Y_cond.drop(columns=['filename_lr']).sum(axis=1)
Y_cond[Y_cond['total_conds'] > 1]
Y_cond['total_conds'].max()

nan

In [32]:
from sklearn.model_selection import train_test_split

Y_s = Y.sample(1000)

#stratified sample

Y_transformed = np.stack(Y_s.indv_diagnostic.values).astype(np.float32)
Y

# Load raw signal data
X = load_raw_data(Y_s, sampling_rate, path)

X = np.transpose(X, (0, 2, 1))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X, Y_transformed, Y_s.index.to_numpy(), test_size=0.1, random_state=56) #stratify=Y_transforme

In [33]:
X_train_torch = torch.tensor(X_train) # change dim ordering for PyTorch
y_train_torch = torch.tensor(y_train)
X_test_torch = torch.tensor(X_test) # change dim ordering for PyTorch
y_test_torch = torch.tensor(y_test)

In [35]:
max_dig = Y.loc[idx_train]['indv_diagnostic_max']

In [56]:
train_dataset = torch.utils.data.TensorDataset(X_train_torch, y_train_torch)
train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

test_dataset = torch.utils.data.TensorDataset(X_test_torch, y_test_torch)
test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=32)


## CNN

In [57]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


class CNet(nn.Module):
  def __init__(self):
    super(CNet,self).__init__()
    #starting with 96

    self.conv11 = nn.Conv1d(12, 64, 9, padding=4)
    self.conv12 = nn.Conv1d(64, 64, 9, padding=4)
    self.bn1 = nn.BatchNorm1d(64)

    self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)

    self.conv21 = nn.Conv1d(64, 128, 9, padding=4)
    self.conv22 = nn.Conv1d(128, 128, 9, padding=4)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv31 = nn.Conv1d(128, 256, 9, padding=4)
    self.conv32 = nn.Conv1d(256, 256, 9, padding=4)
    self.conv33 = nn.Conv1d(256, 256, 9, padding=4)
    self.bn3 = nn.BatchNorm1d(256)

    self.conv41 = nn.Conv1d(256, 512, 9, padding=4)
    self.conv42 = nn.Conv1d(512, 512, 9, padding=4)
    self.conv43 = nn.Conv1d(512, 512, 9, padding=4)
    self.bn4 = nn.BatchNorm1d(512)

    self.fc1 = nn.Linear(512 * 125, 1000)
    self.fc2 = nn.Linear(1000, 500)
    self.fc3 = nn.Linear(500, 44)

  def forward(self, x):
    #x = self.pool(F.relu(self.conv12(F.relu(self.conv11(x)))))
    #x = self.pool(F.relu(self.conv22(F.relu(self.conv21(x)))))
    #x = self.pool(F.relu(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x)))))))
    #x = F.relu(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x))))))

    x = self.pool(F.relu(self.bn1(self.conv12(F.relu(self.conv11(x))))))
    x = self.pool(F.relu(self.bn2(self.conv22(F.relu(self.conv21(x))))))
    x = self.pool(F.relu(self.bn3(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x))))))))
    x = F.relu(self.bn4(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x)))))))
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

In [58]:
def train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    #loss_fn = torch.nn.CrossEntropyLoss()

    #pos_weight = torch.tensor([5.0]).to(device)
    #loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    #condition_freq = y_train.sum(axis=0)
    #pos_weight = (len(y_train) - condition_freq) / condition_freq

    f = torch.tensor(y_train).sum(dim=0)
    N = y_train.shape[0]
    pos_weight = (N - f) / (f + 1e-6)
    pos_weight = pos_weight.clamp(1.0, 20.0).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))

    lr = 1e-2
    opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    #opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for batch_index, (X, y) in enumerate(train_dataloader):
            print(f"Epoch: {epoch} Batch {batch_index}")
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            running_loss += loss.item()

            loss.backward() #compute gradients

            opt.step() #apply gradients
            #scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")


In [40]:
y_train.sum(axis=0)

array([ 30.,   8.,   3.,   1., 110.,  15.,  25.,  16.,  14.,  18.,  13.,
        13.,   7.,   5.,   8.,   6.,   6.,   2.,   3.,   5.,   3.,   2.,
         1.,   0.,   3.,   1.,   0.,   0.,   0.,   1.,   1.,   1.,   1.,
         0.,   0.,   1.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.],
      dtype=float32)

In [59]:
net = CNet()
train_model(net, train_dataloader, test_dataloader, 10, 1e-3)

C:\Users\wkmp2\AppData\Local\Temp\ipykernel_4120\1444015779.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))


Epoch: 0 Batch 0
Epoch: 0 Batch 1
Epoch: 0 Batch 2
Epoch: 0 Batch 3
Epoch: 0 Batch 4
Epoch: 0 Batch 5
Epoch: 0 Batch 6
Epoch: 0 Batch 7
Epoch: 0 Batch 8
Epoch: 0 Batch 9
Epoch: 0 Batch 10
Epoch: 0 Batch 11
Epoch: 0 Batch 12
Epoch: 0 Batch 13
Epoch: 0 Batch 14
Epoch: 0 Batch 15
Epoch: 0 Batch 16
Epoch: 0 Batch 17
Epoch: 0 Batch 18
Epoch: 0 Batch 19
Epoch: 0 Batch 20
Epoch: 0 Batch 21
Epoch: 0 Batch 22
Epoch: 0 Batch 23
Epoch: 0 Batch 24
Epoch: 0 Batch 25
Epoch: 0 Batch 26
Epoch: 0 Batch 27
Epoch: 0 Batch 28
Epoch 0 train loss: 0.8118948874802425
Epoch: 1 Batch 0
Epoch: 1 Batch 1
Epoch: 1 Batch 2
Epoch: 1 Batch 3
Epoch: 1 Batch 4
Epoch: 1 Batch 5
Epoch: 1 Batch 6
Epoch: 1 Batch 7
Epoch: 1 Batch 8
Epoch: 1 Batch 9
Epoch: 1 Batch 10
Epoch: 1 Batch 11
Epoch: 1 Batch 12
Epoch: 1 Batch 13
Epoch: 1 Batch 14
Epoch: 1 Batch 15
Epoch: 1 Batch 16
Epoch: 1 Batch 17
Epoch: 1 Batch 18
Epoch: 1 Batch 19
Epoch: 1 Batch 20
Epoch: 1 Batch 21
Epoch: 1 Batch 22
Epoch: 1 Batch 23
Epoch: 1 Batch 24
Epoch: 1 

In [33]:
def dig_accurate(y_preds, y_ground):
    
    y_preds[:, 4] = 0
    element_wise_eq = y_preds == y_ground

    for r in y_preds[torch.all(element_wise_eq, dim=1).int() == 1]:
        if (r.sum() > 0):
            print(r)
    print(y_preds[torch.all(element_wise_eq, dim=1).int() == 1])

    print(f"Row-wise Accuracy (perfect match): {sum(torch.all(element_wise_eq, dim=1).int())/element_wise_eq.shape[0]}")
    print(f"Element Wise Accuracy: {sum(element_wise_eq.int().flatten(0))/torch.numel(element_wise_eq)}")

In [ ]:
from sklearn.metrics import jaccard_score

def dig_accurate(y_preds, y_ground):
    element_wise_eq = y_preds == y_ground
    print(f"Jaccard_Score: {jaccard_score(y_preds, y_ground)}")
    print(f"Row-wise Accuracy (perfect match): {sum(torch.all(element_wise_eq, dim=1).int())/element_wise_eq.shape[0]}")
    print(f"Element Wise Accuracy: {sum(element_wise_eq.int().flatten(0))/torch.numel(element_wise_eq)}")

In [62]:
dig_accurate((torch.sigmoid(net(X_test_torch)) > 0.5).int(), y_test_torch.int())

Row-wise Accuracy (perfect match): 0.3799999952316284
Element Wise Accuracy: 0.9590908885002136


In [30]:
print(net(X_test_torch))
print((torch.sigmoid(net(X_test_torch)) > 0.5).int())
print(y_test_torch.int())
print(y_test_torch.shape)
print((torch.sigmoid(net(X_test_torch)) > 0.5).int() == y_test_torch.int()) #== torch.ones(10,44)
print(torch.eq(torch.sigmoid(net(X_test_torch) > 0.5).int(), y_test_torch.int()))


tensor([[ -8.3005,  -9.1539,  -7.3819,  ..., -55.5446, -50.3587, -50.0327],
        [  0.8788,  -1.7841, -11.9350,  ..., -26.9933, -26.2082, -25.7442],
        [  0.9288,  -0.3073,  -8.0598,  ..., -20.4700, -20.1690, -19.7708],
        ...,
        [  0.8939,  -0.0792,  -4.3928,  ..., -15.5913, -14.5463, -13.9821],
        [ -0.2386,  -1.5334,  -9.9867,  ..., -22.6837, -22.9940, -22.3574],
        [ -1.0714,   1.1095,  -3.0265,  ..., -12.8796, -12.0230, -10.8175]],
       grad_fn=<AddmmBackward0>)
tensor([[0, 0, 0,  ..., 0, 0, 0],
        [1, 0, 0,  ..., 0, 0, 0],
        [1, 0, 0,  ..., 0, 0, 0],
        ...,
        [1, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 1, 0,  ..., 0, 0, 0]], dtype=torch.int32)
tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]], dtype=torch.int32)
torch.Size([30, 44])
te

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class ResidualBlock1D(nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size=9, dropout=0.1):
    super().__init__()
    padding = kernel_size // 2

    self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
    self.bn1 = nn.BatchNorm1d(out_channels)

    self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
    self.bn2 = nn.BatchNorm1d(out_channels)

    self.dropout = nn.Dropout(dropout)

    if in_channels != out_channels:
      #residual would be uneven here
      self.shortcut = nn.Conv1d(in_channels, out_channels, kernel_size=1) # reduce the channels
    else:
      self.shortcut = nn.Identity()

  def forward(self, x):
    identity = self.shortcut(x) #residual

    x = self.conv1(x)
    x = self.bn1(x)
    x = F.relu(x)
    x = self.dropout(x)

    x = self.conv2(x)
    x = self.bn2(x)
        
    x = x + identity

    x = F.relu(x)

    return x
    

class CINet(nn.Module):
  def __init__(self):
    super(self).__init__()
    self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
    
    self.block1 = ResidualBlock1D(12, 64, kernel_size=9, dropout=0.1)

    self.block2 = ResidualBlock1D(64, 128, kernel_size=9, dropout=0.1)

    self.block3 = ResidualBlock1D(128, 256, kernel_size=9, dropout=0.2)

    self.block4 = ResidualBlock1D(256, 512, kernel_size=9, dropout=0.2)

    self.gap = nn.AdaptiveAvgPool1d(1)
    
    self.drop_fc = nn.Dropout(0.4)
    self.fc1 = nn.Linear(512, 128)
    self.fc2 = nn.Linear(128, 44)

  def forward(self, x):

    x = self.block1(x)
    x = self.pool(x)

    x = self.block2(x)
    x = self.pool(x)

    x = self.block3(x)
    x = self.pool(x)

    x = self.block4(x)

    x = self.gap(x)
    x = x.squeeze(-1)

    x = self.drop_fc(x)
    x = F.relu(self.fc1(x))
    x = self.drop_fc(x)

    x = self.fc2(x)
    return x
    

"""
class CINet(nn.Module):
  def __init__(self):
    super(CNet,self).__init__()
    
    self.drop_conv = nn.Dropout(0.2)
    self.drop_fc = nn.Dropout(0.4)

    self.conv11 = nn.Conv1d(12, 64, 9, padding=4)
    self.conv12 = nn.Conv1d(64, 64, 9, padding=4)
    self.bn1 = nn.BatchNorm1d(64)

    self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)

    self.conv21 = nn.Conv1d(64, 128, 9, padding=4)
    self.conv22 = nn.Conv1d(128, 128, 9, padding=4)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv31 = nn.Conv1d(128, 256, 9, padding=4)
    self.conv32 = nn.Conv1d(256, 256, 9, padding=4)
    self.conv33 = nn.Conv1d(256, 256, 9, padding=4)
    self.bn3 = nn.BatchNorm1d(256)

    self.conv41 = nn.Conv1d(256, 512, 9, padding=4)
    self.conv42 = nn.Conv1d(512, 512, 9, padding=4)
    self.conv43 = nn.Conv1d(512, 512, 9, padding=4)
    self.bn4 = nn.BatchNorm1d(512)

    self.gap = nn.AdaptiveAvgPool1d(1)
    
    self.fc1 = nn.Linear(512, 128)
    self.fc2 = nn.Linear(128, 44)

  def forward(self, x):

    x = self.pool(F.relu(self.bn1(self.conv12(F.relu(self.conv11(x))))))
    x = self.drop_conv(x)

    x = self.pool(F.relu(self.bn2(self.conv22(F.relu(self.conv21(x))))))
    x = self.drop_conv(x)

    x = self.pool(F.relu(self.bn3(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x))))))))
    x = self.drop_conv(x)

    x = F.relu(self.bn4(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x)))))))
    x = self.gap(x)
    x = x.squeeze(-1)
    x = x.drop_fc(x)
    x = F.relu(self.fc1(x))
    x = x.drop_fc(x)
    x = self.fc2(x)
    return x"""